In [1]:
# BEAD THREADING TEST - FULL BIO PIPELINE

import cv2
import numpy as np
import mediapipe as mp
from scipy.signal import find_peaks
import math


In [2]:
# MediaPipe Setup (Hands + Pose)

mpHands = mp.solutions.hands
hands = mpHands.Hands(min_detection_confidence=0.6)

mpPose = mp.solutions.pose
pose = mpPose.Pose(min_detection_confidence=0.6)

mpDraw = mp.solutions.drawing_utils

DRAWING_SPEC_LANDMARK = mpDraw.DrawingSpec(color=(0,0,255), thickness=2, circle_radius=2)
DRAWING_SPEC_CONNECTION = mpDraw.DrawingSpec(color=(0,0,0), thickness=2)

In [3]:
# Utility Functions

def safe_mean(arr, default=0.0):
    return sum(arr)/len(arr) if len(arr) > 0 else default

def safe_std(arr, default=0.0):
    return float(np.std(arr)) if len(arr) > 0 else default

In [4]:
# HAND STABILITY (TREMOR)

def hand_stability(x, y):
    return safe_std(x) + safe_std(y)

In [5]:
# RHYTHM + COORDINATION
def compute_metrics(dom_y, sup_y, time_list):

    dom_y = np.array(dom_y)
    sup_y = np.array(sup_y)
    t = np.array(time_list)

    # -------- RHYTHM --------
    peaks, _ = find_peaks(-dom_y, distance=3)
    if len(peaks) > 1:
        rhythm = safe_std(np.diff(t[peaks]), 0.2)
    else:
        rhythm = 0.2

    # -------- COORDINATION --------
    min_len = min(len(dom_y), len(sup_y))
    sync = safe_mean(np.abs(dom_y[:min_len] - sup_y[:min_len]))

    return rhythm, sync

In [6]:
def primary_score_cal(bead_count):
    if bead_count >= 30:
        return 5
    elif bead_count >= 24:
        return 4
    elif bead_count >= 18:
        return 3
    elif bead_count >= 12:
        return 2
    else:
        return 1

In [7]:
# ERROR ADJUSTMENT
def adjusted_score(base_score, errors):
    if errors <= 2:
        return base_score
    elif errors <= 5:
        return base_score - 0.5
    elif errors <= 8:
        return base_score - 1
    else:
        return min(base_score, 2)

In [8]:
# QUALITY SCORE (BIO)
def quality_score(stability, rhythm, sync, accel_var, jerk_var, fatigue):

    quality = 0
    # Stability
    if stability < 0.02: quality += 2
    elif stability < 0.05: quality += 1

    # Rhythm
    if rhythm < 0.05: quality += 2
    elif rhythm < 0.08: quality += 1

    # Coordination
    if sync < 0.02: quality += 2
    elif sync < 0.05: quality += 1

    # Fluidity
    if accel_var < 0.04 and jerk_var < 0.1: quality += 1

    # Fatigue
    if fatigue < 0.02: quality += 1

    return quality

In [9]:
# FINAL SCORE
def final_score_cal(base_score, quality):
    if base_score >= 4 and quality >= 6:
        return 5
    elif base_score >= 3 and quality >= 4:
        return 4
    elif base_score >= 3:
        return 3
    elif base_score == 2:
        return 2
    else:
        return 1

In [10]:
def category_cal(final_score):
    if final_score == 5:
        return "Excellent"
    elif final_score == 4:
        return "Above Average"
    elif final_score == 3:
        return "Average"
    elif final_score == 2:
        return "Below Average"
    else:
        return "Poor"

In [11]:
# MAIN FUNCTION
def bead_threading(ID="ID001", name="Test", path="video.mp4"):

    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS)

    dom_x, dom_y = [], []
    sup_x, sup_y = [], []
    time_list = []

    bead_count = 0
    errors = 0

    frame_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        hand_results = hands.process(rgb)
        pose_results = pose.process(rgb)

        # DRAW HAND LANDMARKS
        # -------------------------------
        if hand_results.multi_hand_landmarks:
            for hand_landmarks in hand_results.multi_hand_landmarks:
                mpDraw.draw_landmarks(frame, hand_landmarks, mpHands.HAND_CONNECTIONS, DRAWING_SPEC_LANDMARK, DRAWING_SPEC_LANDMARK)

                x = hand_landmarks.landmark[8].x
                y = hand_landmarks.landmark[8].y

                # Assume dominant hand = right side
                if x > 0.5:
                    dom_x.append(x)
                    dom_y.append(y)

                    # THREADING DETECTION (downward precision motion)
                    if y > 0.6:
                        bead_count += 1

                else:
                    sup_x.append(x)
                    sup_y.append(y)

        # DRAW POSE (OPTIONAL BUT INCLUDED)
        if pose_results.pose_landmarks:
            mpDraw.draw_landmarks(frame, pose_results.pose_landmarks, mpPose.POSE_CONNECTIONS, DRAWING_SPEC_LANDMARK, DRAWING_SPEC_CONNECTION)

        window_name = "Bead Threading Analysis"
        cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)
        cv2.putText(frame, f"Frame: {frame_idx}  Beads: {bead_count}", (20, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2,)
        cv2.imshow(window_name, frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

        if hand_results.multi_hand_landmarks:
            time_list.append(frame_idx / fps if fps else 0)
        frame_idx += 1

    cap.release()
    cv2.destroyAllWindows()

    # =========================================================
    # BIOLOGICAL METRICS
 
    rhythm, sync = compute_metrics(dom_y, sup_y, time_list)

    # -------- STABILITY --------
    dom_stability = hand_stability(dom_x, dom_y)
    sup_stability = hand_stability(sup_x, sup_y)
    stability = (dom_stability + sup_stability) / 2

    # -------- FATIGUE --------
    mid = len(dom_y) // 2
    fatigue = abs(safe_mean(dom_y[:mid]) - safe_mean(dom_y[mid:]))

    # -------- FLUIDITY --------
    if len(dom_y) > 3:
        vel = np.diff(dom_y)
        accel = np.diff(vel)
        jerk = np.diff(accel)
        accel_var = safe_std(accel, 1.0)
        jerk_var = safe_std(jerk, 1.0)
    else:
        accel_var = 1.0
        jerk_var = 1.0


    # PRIMARY SCORE (BEAD COUNT)
    base_score = primary_score_cal(bead_count)

    # ERROR ADJUSTMENT
    base_score = adjusted_score(base_score, errors)

    # QUALITY SCORE (BIO)
    quality = quality_score(stability, rhythm, sync, accel_var, jerk_var, fatigue)


    # FINAL SCORE
    final_score = final_score_cal(base_score, quality)

    #category
    category = category_cal(final_score)

    # =========================================================
    # OUTPUT
    print("------ BEAD THREADING RESULT ------")
    print(f"Beads Threaded: {bead_count}")
    print(f"Stability: {stability:.3f}")
    print(f"Rhythm: {rhythm:.3f}")
    print(f"Coordination: {sync:.3f}")
    print(f"Fatigue: {fatigue:.3f}")
    print(f"Accel: {accel_var:.3f}, Jerk: {jerk_var:.3f}")
    print(f"Final Score: {final_score}")
    print(f"Category: {category}")

    return final_score, category

In [13]:
path = "data/beads_FM_2.mp4"
bead_threading(ID="ID001", name="Test", path=path)

------ BEAD THREADING RESULT ------
Beads Threaded: 153
Stability: 0.233
Rhythm: 0.245
Coordination: 0.107
Fatigue: 0.081
Accel: 0.304, Jerk: 0.606
Final Score: 3
Category: Average


(3, 'Average')